# 01 · Extração: SQL Server → Databricks `landing/dados` (CSV)

**Trabalho 3 — Lakehouse com Databricks e arquitetura medalhão** · Autores: **Murilo Mandelli** e **Augusto Benedetti**

Roda **no computador (WSL)**. Extrai **todas as tabelas** do banco `LojaDB` para **CSV** e envia
cada arquivo para o **volume `dados` do schema `landing`** no Databricks
(`/Volumes/workspace/landing/dados/`).

> **Por que esta etapa roda fora do Databricks?** O Databricks Free Edition roda na nuvem e não
> enxerga um banco que está no nosso computador (`localhost`). Por isso a extração é feita
> localmente e os arquivos são enviados para a nuvem — exatamente o papel de uma ferramenta de
> ingestão. A partir da landing, tudo roda no Databricks, encadeado pela **Job**.

**Pré-requisito:** notebook `00` executado.

In [1]:
from io import BytesIO

import pandas as pd

from lakehouse_databricks import (
    cliente_databricks,
    conectar_sqlserver,
    databricks_configurado,
    garantir_volume_landing,
)
from lakehouse_databricks.config import CAMINHO_VOLUME, DB_DATABASE, PASTA_LANDING

conn = conectar_sqlserver(DB_DATABASE)
cursor = conn.cursor()
PASTA_LANDING.mkdir(parents=True, exist_ok=True)
print("Destino local :", PASTA_LANDING)
print("Destino nuvem :", CAMINHO_VOLUME)

Destino local : /root/lakehouse-databricks/data/landing
Destino nuvem : /Volumes/workspace/landing/dados


## 1. Descobrir todas as tabelas do banco (`INFORMATION_SCHEMA`)

In [2]:
cursor.execute("""
SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = 'dbo'
ORDER BY TABLE_NAME
""")
tabelas = [linha[0] for linha in cursor.fetchall()]
print(tabelas)

['categoria', 'cliente', 'item_pedido', 'pedido', 'produto']


## 2. Extrair cada tabela para CSV

In [3]:
resumo = []
for tabela in tabelas:
    cursor.execute(f"SELECT * FROM [{tabela}]")
    colunas = [c[0] for c in cursor.description]
    df = pd.DataFrame.from_records(cursor.fetchall(), columns=colunas)
    caminho = PASTA_LANDING / f"{tabela}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8")
    resumo.append({"arquivo": caminho.name, "registros": len(df), "tamanho_kb": round(caminho.stat().st_size / 1024, 1)})
cursor.close()
conn.close()
pd.DataFrame(resumo)

,arquivo,registros,tamanho_kb
0,categoria.csv,8,0.4
1,cliente.csv,1015,82.7
2,item_pedido.csv,8971,192.1
3,pedido.csv,3000,131.5
4,produto.csv,200,7.8


## 3. Enviar os CSV para o volume `landing/dados` no Databricks

Se o `.env` tiver `DATABRICKS_HOST` e `DATABRICKS_TOKEN`, o envio é automático pelo
**Databricks SDK**: cria o schema `landing` e o volume `dados` (se não existirem) e faz o upload.

Sem token, faça o upload pela interface: **Catalog ▸ workspace ▸ landing ▸ dados ▸ Upload to
this volume** e selecione os 5 arquivos da pasta `data/landing/`.

In [4]:
if databricks_configurado():
    w = cliente_databricks()
    garantir_volume_landing(w)
    for item in resumo:
        with open(PASTA_LANDING / item["arquivo"], "rb") as arquivo:
            w.files.upload(f"{CAMINHO_VOLUME}/{item['arquivo']}", BytesIO(arquivo.read()), overwrite=True)
        print(f"enviado: {CAMINHO_VOLUME}/{item['arquivo']}")
    print()
    for f in w.files.list_directory_contents(CAMINHO_VOLUME):
        print(f"no volume: {f.name:<18} {f.file_size:>8} bytes")
else:
    print("Databricks não configurado no .env.")
    print(f"Faça o upload manual dos arquivos de {PASTA_LANDING} para {CAMINHO_VOLUME}")

Databricks não configurado no .env.
Faça o upload manual dos arquivos de /root/lakehouse-databricks/data/landing para /Volumes/workspace/landing/dados
